<a href="https://colab.research.google.com/github/snr-laboratory/snrlab-ic-q-pix-v1/blob/main/dev_journals/kgosine_202509_c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


|Pin Name | Description | Current Draw|
|---|---|---|
|VDD| Power supply | |
|VSS|Chip GND| |
|PIXEL_PAD| Charge collection pixel|5nA|
|INT_SPY| Integrator output spy point|$\pm$ 2uA|
|SPY_CURR| Current source for spy point|4uA|
|REPL_OUTP| Positive LVDS signal of calls for replenishment|$\pm$ 3.5mA|
|REPL_OUTN| Negative LVDS signal of calls for replenishment|$\pm$ 3.5mA|
|CLK_INP| Positive LVDS clock signal||
|CLK_INN| Negative LVDS clock signal||
|BIAS_CTRL| Bias current for biasing circuitry||
|VREF| Integrator threshold voltage||
|VREF_COMP| Comparator threshold voltage||


Option for Clk: https://www.mouser.com/datasheet/3/282/1/DSC1103-23-Low-Jitter-Precision-LVDS-Oscillator-DS20005745C.pdf


Useful links for the comparator design:
https://ieeexplore.ieee.org/document/4242391
https://picture.iczhiku.com/resource/ieee/WHiYwoUjPHwZPXmv.pdf


### The problem with the pad capacitance on the circuit bandwidth

No 3pF cap to simulate pixel pad on the input to the circuit
[Layout opamp, layout comparator, layout bias circuit, schematic replenishment circuit (1.8V/18MOhm)]

![](https://drive.google.com/uc?export=view&id=1M13t9tcAiP43KLz_Lwm2oAb9ZWDU9zX-)

No problem, the input is 2fC, the replenishments add up to about the same.

But when I add the 3pF cap, the slow response of the opamp to replenishments causes the opamp output to be above the comparator threshold for longer, calling for additional replenishments (almost twice as many):

![](https://drive.google.com/uc?export=view&id=1pExmA5IKAuoJUoDX29aEg0S425Lb2JWw)


Testing the problem with the realitic current signal (this is with the 3pF cap):

![](https://drive.google.com/uc?export=view&id=1mrkS8uTD0Z6QYpi8-e7GWoqnU9wHxacf)

You can see the problem is really for sharp current peaks that the bandwidth limit causes extra replenishments (see the 2nd and 3rd spikes where the integrator output falls below baseline=900mV)

I'm fine with this for now since the reconstruction of current arriving is still good in this case:

![](https://drive.google.com/uc?export=view&id=1jbI61ka4pXvsuB12xTIEy3X2ypvAu1na)
![](https://drive.google.com/uc?export=view&id=1fqf-GXV94hSDz6Xurr81yTljykMEd6eo)

You see the reconstructed charge is larger than the actual charge during those large spikes.

### Attempts to reduce size of replenishment resistor

Using the 1.8V Vdd for the replenishment circuit into the 900mV integrator input ($\Delta $V = 900mV) for $\Delta$ Q= 0.25fC in $10ns$ means $I= 25nA$ so $R=\frac{900mV}{25nA}= 36 M \Omega $. To reduce the size of this resistor we can first increase $\Delta Q$ from $0.25fC →0.5 fC$. We can also reduce the high voltage in the circuit from $1.8V \rightarrow  1.2V$ ($\Delta V$ = 300mV instead of 900mV). Then the resistor needs to be $R= \frac{300mV}{50nA}= 6M\Omega$.

![](https://drive.google.com/uc?export=view&id=1v6szzI-noWleLmiT_maDLl16cSXYbFNH)

![](https://drive.google.com/uc?export=view&id=1zK3k_oq7FuNkScVBW18chWkDeT3pl8cV)

Note that no 3pF input capacitor was connected. This is with the layout opamp, layout comparator, layout biasing circuitry, schematic modified replenishment circuit.


Going to lower voltages across the resistor reduces the fidelity of the replenishment signals.


![](https://drive.google.com/uc?export=view&id=1jPh3ZQWFX_f7y4txmeyoh7iTgXic4Uzh)

![](https://drive.google.com/uc?export=view&id=10R9uzI9xMsLXv8GYrCyotk0e7El4BC4U)



Not directly connecting the replenishment circuit to Vdd and instead some other voltage (called Avdd for now) would also make the replenishment charge variable between 0.5fC and 1.5fC by varying Avdd between 1.2V upto 1.8V. This could be further improveed if we considered operating the opamp at a DC voltage less than 900mV. I didn't really investigate this and since the circuit only functions in one direction (opamp output risng from its DC value) there are possible benefits in lowering this value.



### Post-Layout Simulation of Replenishment Circuit

Laid out the 1.2V, 6MOhm option. Went with 60 segments so each segment width was just under 300um. Simulating with 2fC test pulse, no 3pF input capacitor, layedout bias circuitry, opamp, comparator, and replenishment circuit:


![](https://drive.google.com/uc?export=view&id=1tWR95QnbVjGXHesHn0DpX7Cs08Tch6Ea)

![](https://drive.google.com/uc?export=view&id=1PZhd3T-iegNxPQnRZpDQWeGN72IxBQum)

![](https://drive.google.com/uc?export=view&id=17SNmQ7B-o4vWFefjwa1j2hPWbVlsdbgm)

Lots of spiking on clock edges but integrating the current, they don't significantly contribute to replenished charge.


### Post Layout simulation with realistic charge input & 3pF input capacitor


![](https://drive.google.com/uc?export=view&id=1uE70LgVxBgu7DEV8dY0_Tu2l3uwjjS0v)

![](https://drive.google.com/uc?export=view&id=1daMJmpeZZIOP_Dtr9OfzCVccwHcu9vaU)



### Post Layout Simulation @-40*C

Same setup as previous but simulated at -40*C. Maintained good reconstruction of charge deposited with 0.5fC replenishments. The excessive replenishments in response to sharp input current peaks is actually reduced in the cold simulation (see integrator output dipsto 891mV instead of 886mV in prev trace).  

![](https://drive.google.com/uc?export=view&id=1PDZyqkBBD84lp68BX8Rr0K5npCZL_yvO)

![](https://drive.google.com/uc?export=view&id=1O6_vVcRYSPmNPVCKNgm5AeTBN0hErq9-)


## Spy Point

![](https://drive.google.com/uc?export=view&id=1wupAPGPQVhwP9tewq0GL3CoWMsrfRa6L)

![](https://drive.google.com/uc?export=view&id=1V0-396r49TRBHtbvqPHMhYa-k9XOhBLr)

![](https://drive.google.com/uc?export=view&id=114TXgRcOeQ_QFHPhiloiKYBq6qJZ-vVU)

The external voltage is 1.25V across a 10KOhm resistor. The spy_curr and spy_out would be separate pads. The all the transistors are L=230nm and the PMOS_SF_Width=6um, the CurrentMirror_widths=4u for both.


### Spy point post layout simulation

The DC output voltage is 1.479V, the DC current draw is 3.719uA.
The gain is slightly increased to 0.756.

![](https://drive.google.com/uc?export=view&id=1svbty6jR48TtJQ-H1uMSfFRhgqd088Ud)

![](https://drive.google.com/uc?export=view&id=13uLw18k6TB3jrkQibgGy1ijBMTbYI945)


### LVDS Transmitter

![](https://drive.google.com/uc?export=view&id=1xPusvKe-Zi7_7bJGmuiKEGxXL-6VKhDf)

![](https://drive.google.com/uc?export=view&id=1BCMyiXoMsoUCctYieu3pMN-iLQjMlW-i)

![](https://drive.google.com/uc?export=view&id=15O_xiG3VC30EpGhg677WR7b9ynh8B8Ij)

Working with about 3.5mA current & 100Ohm termination resistor.
Still need to figure out the bottom current source input.

### LVDS Reciever


![](https://drive.google.com/uc?export=view&id=1M_3-JNHUqBnoisU1VRepNjm77yzkoxCW)

![](https://drive.google.com/uc?export=view&id=1Re5vdVYmoUIPRsGA1g4FmdTqbzfiKGvP)

Went with essentially minimum sizing (larger for PMOS leg) since I'm primarily concerned with speed. Need to layout to really say anything about how well this works.